# ComfyUI in Colab (with a public tunnel)

Runs a full ComfyUI server on the Colab GPU and exposes it through a Cloudflare quick tunnel, so you can open the node editor in your browser and load any ComfyUI workflow template (drag the `.json` onto the canvas).

1. Run the install cell (clones ComfyUI and installs its requirements).
2. Run the model cell — it downloads an SD1.5 checkpoint so there is something to generate with. Add more checkpoints to `ComfyUI/models/checkpoints`, LoRAs to `models/loras`.
3. Run the last cell and open the `https://….trycloudflare.com` link it prints. Keep that cell running while you work.

Part of the SoapBox GenAI school (hathor.soapbox.community/colab). Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt

In [ ]:
!wget -q -c https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors -P /content/ComfyUI/models/checkpoints/
!ls -lh /content/ComfyUI/models/checkpoints/

In [ ]:
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
import subprocess, threading, time, socket, re

def wait_for_port(port, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(1)
    return False

def tunnel(port):
    if not wait_for_port(port):
        print("ComfyUI did not start — check the log above."); return
    p = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("\n>>> Open ComfyUI at:", m.group(0), "\n")

threading.Thread(target=tunnel, args=(8188,), daemon=True).start()
!cd /content/ComfyUI && python main.py --listen 127.0.0.1 --port 8188